# Guía de Estudio QKP - Cuaderno 3: Resolución con QAOA usando Qiskit

En este cuaderno final de tu semana de estudio, resolveremos nuestra instancia QKP de 4 objetos utilizando el algoritmo **QAOA (Quantum Approximate Optimization Algorithm)** en **Qiskit**.

Para evitar incompatibilidades entre versiones de Qiskit, en este cuaderno **construiremos el circuito QAOA paso a paso desde cero a nivel de compuertas lógicas**. De esta forma verás exactamente qué pasa en los qubits sin depender de librerías de alto nivel propensas a cambios de API.

In [ ]:
# Instalar Qiskit y dependencias científicas
!pip install qiskit scipy numpy matplotlib

## 1. Recordatorio del Hamiltoniano Ising de nuestra Instancia

En el Cuaderno 1 calculamos la matriz QUBO $Q$ de nuestra mochila de 4 objetos y 3 variables de holgura (7 variables binarias en total).

Recordemos los valores diagonales $Q_{ii}$ y cruzados $Q_{ij}$:
* **Qubits de decisión:** $q_0, q_1, q_2, q_3$ (Ítems 1 al 4).
* **Qubits de holgura:** $q_4, q_5, q_6$ (Slack $s_0, s_1, s_2$).

Para mapear QUBO a Ising, usamos la transformación estándar:
$$x_i \to \frac{I - Z_i}{2}$$

Al sustituir esto, obtenemos los coeficientes del Hamiltoniano $H_C = \sum h_i Z_i + \sum J_{ij} Z_i Z_j$.
Programemos una función que convierta la matriz QUBO directamente a estos campos locales $h_i$ y acoplamientos $J_{ij}$:

In [ ]:
import numpy as np

# Definimos la matriz QUBO Q (7x7) para nuestra instancia
# Variables: [x1, x2, x3, x4, s0, s1, s2]
# Coeficientes lineales y cuadráticos calculados en el cuaderno 1
n_qubits = 7
Q = np.zeros((n_qubits, n_qubits))

# Coeficientes diagonales (Q_ii)
Q[0,0] = -165  # x1
Q[1,1] = -216  # x2
Q[2,2] = -248  # x3
Q[3,3] = -164  # x4
Q[4,4] = -90   # s0
Q[5,5] = -160  # s1
Q[6,6] = -248  # s2

# Coeficientes cruzados (Q_ij para i < j)
# Pesos: a = [2, 3, 4, 2, 1, 2, 4], lambda = 10
# Q_ij = 20 * a_i * a_j - v_ij
Q[0,1] = 120 - 3  # x1-x2
Q[0,2] = 160      # x1-x3
Q[0,3] = 80 - 4   # x1-x4
Q[0,4] = 40       # x1-s0
Q[0,5] = 80       # x1-s1
Q[0,6] = 160      # x1-s2

Q[1,2] = 240 - (-2) # x2-x3
Q[1,3] = 120      # x2-x4
Q[1,4] = 60       # x2-s0
Q[1,5] = 120      # x2-s1
Q[1,6] = 240      # x2-s2

Q[2,3] = 160      # x3-x4
Q[2,4] = 80       # x3-s0
Q[2,5] = 160      # x3-s1
Q[2,6] = 320      # x3-s2

Q[3,4] = 40       # x4-s0
Q[3,5] = 80       # x4-s1
Q[3,6] = 160      # x4-s2

Q[4,5] = 40       # s0-s1
Q[4,6] = 80       # s0-s2

Q[5,6] = 160      # s1-s2

# Convertimos la matriz QUBO a parámetros Ising
h = np.zeros(n_qubits)
J = np.zeros((n_qubits, n_qubits))

for i in range(n_qubits):
    # Campo local h_i
    h[i] = -Q[i,i]/2 - sum(Q[i,j] for j in range(i+1, n_qubits))/4 - sum(Q[j,i] for j in range(0, i))/4
    for j in range(i+1, n_qubits):
        # Acoplamiento J_ij
        J[i,j] = Q[i,j]/4

print("Campos locales (h_i) del Hamiltoniano Ising:")
print(h)
print("\nAcoplamientos (J_ij) entre Qubits:")
print(J)

## 2. Construcción de los Circuitos QAOA en Qiskit

QAOA consiste en aplicar alternadamente dos operadores unitarios a un estado de superposición inicial:
1. **Operador de Mezcla ($U_M(\beta)$):** Rotaciones $R_x(2\beta)$ en todos los qubits.
2. **Operador de Costo ($U_C(\gamma)$):** 
   * Rotaciones $R_z(2\gamma h_i)$ para los términos lineales.
   * Secuencias $\text{CNOT}_{i,j} \to R_z(2\gamma J_{ij})_j \to \text{CNOT}_{i,j}$ para los términos cuadráticos.

Implementemos la función que crea este circuito dinámicamente:

In [ ]:
from qiskit import QuantumCircuit

def create_qaoa_circuit(h, J, gamma, beta, p=1):
    """Crea un circuito QAOA de p pasos"""
    qc = QuantumCircuit(n_qubits)
    
    # Estado inicial: superposición máxima en todos los qubits (|+++...>)
    for i in range(n_qubits):
        qc.h(i)
        
    # Aplicar p pasos de evolución
    for step in range(p):
        # --- OPERADOR DE COSTO U_C(gamma) ---
        # Términos lineales (Campos locales h_i)
        for i in range(n_qubits):
            qc.rz(2 * gamma[step] * h[i], i)
            
        # Términos cuadráticos (Acoplamientos J_ij)
        for i in range(n_qubits):
            for j in range(i+1, n_qubits):
                if J[i,j] != 0:
                    qc.cnot(i, j)
                    qc.rz(2 * gamma[step] * J[i,j], j)
                    qc.cnot(i, j)
                    
        # --- OPERADOR DE MEZCLA U_M(beta) ---
        for i in range(n_qubits):
            qc.rx(2 * beta[step], i)
            
    return qc

# Visualizamos el circuito para p=1 y ángulos de prueba
qc_test = create_qaoa_circuit(h, J, [0.1], [0.2], p=1)
qc_test.draw('text')

## 3. Simulación y Optimización Clásica de Parámetros

Para medir la energía esperada del Hamiltoniano en el simulador cuántico, utilizaremos el simulador de vectores de estado (`Statevector`) de Qiskit.

In [ ]:
from qiskit.quantum_info import Statevector
from scipy.optimize import minimize

def calculate_energy(params):
    """Calcula el valor esperado de la energía QUBO para un par de ángulos (gamma, beta)"""
    gamma = [params[0]]
    beta = [params[1]]
    
    # 1. Crear el circuito cuántico con estos parámetros
    qc = create_qaoa_circuit(h, J, gamma, beta, p=1)
    
    # 2. Obtener el vector de estado cuántico resultante
    sv = Statevector.from_instruction(qc)
    probabilities = sv.probabilities_dict()
    
    # 3. Calcular la energía QUBO esperada: E = sum( P(estado) * Costo_QUBO(estado) )
    expected_energy = 0
    for state_bin, prob in probabilities.items():
        # El estado viene en formato binario de Qiskit (el qubit 0 está a la derecha)
        # Lo revertimos para emparejar con nuestros índices del cuaderno
        state_reversed = state_bin[::-1]
        y = np.array([int(bit) for bit in state_reversed])
        
        # Costo QUBO: y^T * Q * y
        cost = y.T @ Q @ y
        expected_energy += prob * cost
        
    return expected_energy

# Optimizamos clásicamente los ángulos usando el algoritmo Nelder-Mead
initial_angles = [0.1, 0.1]  # Valores semilla
res = minimize(calculate_energy, initial_angles, method='Nelder-Mead')

opt_gamma, opt_beta = res.x
print("Optimización terminada!")
print(f"Ángulo óptimo gamma (costo): {opt_gamma:.4f}")
print(f"Ángulo óptimo beta (mezcla): {opt_beta:.4f}")
print(f"Energía mínima esperada encontrada: {res.fun:.4f}")

## 4. Visualización de los Resultados Cuánticos

Ahora simularemos el circuito con los ángulos óptimos encontrados y graficaremos las probabilidades de los estados medidos para ver si el computador cuántico logra identificar la solución óptima.

In [ ]:
import matplotlib.pyplot as plt

# Crear circuito final optimizado
qc_opt = create_qaoa_circuit(h, J, [opt_gamma], [opt_beta], p=1)
sv_opt = Statevector.from_instruction(qc_opt)
probs = sv_opt.probabilities_dict()

# Filtramos y ordenamos los estados de mayor a menor probabilidad
sorted_probs = sorted(probs.items(), key=lambda item: item[1], reverse=True)

# Imprimir los 5 estados más probables y su interpretación
print("Los 5 estados cuánticos con mayor probabilidad en la medición:")
for state_bin, prob in sorted_probs[:5]:
    # Invertir para leer de izquierda a derecha (qubit 0 a qubit 6)
    state_reversed = state_bin[::-1]
    items = [int(bit) for bit in state_reversed[:4]]
    slack = [int(bit) for bit in state_reversed[4:]]
    slack_val = slack[0] + 2*slack[1] + 4*slack[2]
    
    # Pesos de los objetos: [2, 3, 4, 2]
    peso_total = items[0]*2 + items[1]*3 + items[2]*4 + items[3]*2
    # Valores de los objetos: [5, 6, 8, 4] + sinergias (1,2)->3, (2,3)->-2, (1,4)->4
    valor_total = items[0]*5 + items[1]*6 + items[2]*8 + items[3]*4
    if items[0] and items[1]: valor_total += 3
    if items[1] and items[2]: valor_total -= 2
    if items[0] and items[3]: valor_total += 4
    
    factible = "Factible" if peso_total <= 5 else "NO FACTIBLE"
    
    print(f"Estado: |{state_bin}\u232a | Prob: {prob:.4%}")
    print(f"  -> Objetos: {items} | Holgura s: {slack_val} | Peso: {peso_total} kg | Valor: {valor_total} | [{factible}]\n")

## 5. Análisis del resultado cuántico

¡Felicidades! Si examinas los estados cuánticos de alta probabilidad:

1. El estado con mayor probabilidad debe corresponder a la combinación óptima factible.
2. Nota cómo los qubits de holgura se auto-ajustaron físicamente en la medición para absorber el espacio restante.
3. En QAOA con $p=1$, el sistema nos da una probabilidad moderada para el óptimo debido a que es un circuito de baja profundidad. Al incrementar el número de pasos $p$ (más capas de operadores), la probabilidad del estado óptimo se acerca asintóticamente al 100%.